# Inference test — final DPO model
Loads the final DPO-aligned adapter and runs it against real questions via
a `generate_answer(question)` function. This is the project's inference
test: not one of the 3 required training notebooks, just proof the final
model actually answers correctly end to end.

For an interactive Base vs SFT vs DPO comparison dashboard (CPU-only, no
GPU needed), see `entire_ft_summary_in_oneshot.ipynb` in the project root
instead.



In [ ]:
# mount Drive and locate the project
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'

if not os.path.exists(f'{PROJECT}/adapters/stage3_dpo/adapter_config.json'):
    raise FileNotFoundError(
        "adapters/stage3_dpo is missing -- run notebooks/dpo_alignment.ipynb "
        "(Stage 3) all the way through first, this test needs that adapter."
    )
print('Found DPO adapter at:', f'{PROJECT}/adapters/stage3_dpo')

Mounted at /content/drive
Found DPO adapter at: /content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage3_dpo


In [ ]:
# fail fast if there's no GPU, with clear instructions
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected in this runtime.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, "
        "then Runtime -> Restart session, then run all cells again from the top."
    )
print("GPU OK:", torch.cuda.get_device_name(0))

GPU OK: Tesla T4


In [ ]:
%%capture
# Cell -- install Unsloth
!pip install --no-deps unsloth
!pip install unsloth_zoo
!pip install --no-deps peft accelerate bitsandbytes xformers

## Load the model and define generate_answer()


In [ ]:
# load the final DPO adapter and define generate_answer()
from unsloth import FastLanguageModel

MODEL_PATH = f"{PROJECT}/adapters/stage3_dpo"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=2048,
    dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def generate_answer(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("Model loaded, generate_answer() ready.")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded, generate_answer() ready.


In [ ]:
# the actual test
question = "How can I apply for a refund?"
answer = generate_answer(question)
print("Q:", question)
print("A:", answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I apply for a refund?
A: I'm here to assist you with applying for a refund! Rest assured that we take your satisfaction very seriously and want to make sure you have an easy and seamless process. To initiate the refund application, please follow these steps:

1. Log in to your account on our website at support.example.com using your credentials.
2. Navigate to the "Order History" or "Order History" section within your account dashboard.
3. Locate the specific order associated with the product or service you would like to request a refund for.
4. Click on the order number or reference number to access the details of the order.
5. Look for the option labeled "Refund" or "Refunds" and select it.
6. Fill out any additional information or questions related to the refund as prompted.
7. Provide feedback about why you need a refund and if there's anything else we can do to help during this process.

If you encounter any difficulties or have further questions along the way, feel fre

In [ ]:
# try a couple more, just to be thorough
for q in [
    "How do I cancel an order I just placed?",
    "The item I received doesn't match the description, what can I do?",
]:
    print("Q:", q)
    print("A:", generate_answer(q))
    print("---")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I cancel an order I just placed?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: We understand your desire to cancel the order you recently placed. To proceed with the cancellation, please follow these steps:

1. Access Your Account: Log in to your account portal at support.example.com using your credentials.

2. Locate Order History: Once logged in, navigate to the "Order History" or "History" section of your account.

3. Find the Specific Order: Look for and select the order associated with the purchase number #SC-26745.

4. Initiate Cancellation: Within the selected order details page, locate the option labeled "Order History" or "Cancellation." Click on it to initiate the cancellation process.

5. Follow Any Additional Instructions: The system may provide additional prompts or questions during the cancellation process. Please respond as needed.

6. Confirm Cancellation: After completing any required actions, review the confirmation message or status update to ensure that the order has been successfully canceled.

If you encounter any difficulties or have fur

## Result
If the cells above printed real, sensible, on-topic answers with no errors --
the final DPO model and `generate_answer()` are proven to work end to end.